# Fall Detection: IMU Classifier Training
This notebook trains a neural network to distinguish between "FALL" and "NON-FALL" events using 1 second of IMU data (Accel/Gyro).

### 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix

# Parameters from IMU_Fall_Capture.ino
SAMPLES_PER_GESTURE = 119
NUM_CHANNELS = 6 # ax, ay, az, gx, gy, gz
CLASSES = ['fall', 'normal']

### 2. Model Architecture
We use a Dense (MLP) network for motion classification, which is lightweight for the Nano 33.

In [ ]:
# (Assuming train_data, train_labels, val_data, val_labels are prepared)
model = models.Sequential([
    layers.Input(shape=(SAMPLES_PER_GESTURE * NUM_CHANNELS,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

history = model.fit(train_data, train_labels, validation_data=(val_data, val_labels), epochs=100)

### 3. Evaluating the Model (Error Tracking)
This section tracks the training progress and helps you detect if the model is overfitting (e.g., if it only recognizes *your* specific fall).

In [ ]:
# Plot Training & Validation Accuracy/Loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('IMU Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('IMU Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

### 4. Confusion Matrix
This heatmap shows how many 'Normal' movements (like walking) were incorrectly flagged as 'Falls'.

In [ ]:
y_pred = np.argmax(model.predict(test_data), axis=1)
cm = confusion_matrix(test_labels, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Fall Detection Confusion Matrix')
plt.show()